In [1]:
import pandas as pd
import librosa
import numpy as np

In [8]:
audio_path = "audio_data/STOLE_YA_FLOW_ASAP_ROCKY.mp3"
print("Loading audio file... (this may take a few seconds)")
y, sr = librosa.load(audio_path)

Loading audio file... (this may take a few seconds)


In [5]:
# 2. Extract Tempo (BPM)
# Librosa analyzes the onset strengths of percussion to calculate the beat timing
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
# Handle array unpacking if librosa returns an array for tempo
bpm = tempo[0] if isinstance(tempo, (np.ndarray, list)) else tempo
print(f"Estimated BPM: {bpm:.2f}")

# 3. Extract Chromagram (Key & Chord DNA)
# This maps the audio signal down to the 12 semitones of the musical scale
chroma = librosa.feature.chroma_stft(y=y, sr=sr)
mean_chroma = np.mean(chroma, axis=1)

# Print out the intensity of each pitch class (C, C#, D, etc.)
pitch_classes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
print("\nChroma Profile (Pitch Intensities 0.0 to 1.0):")
for note, intensity in zip(pitch_classes, mean_chroma):
    print(f"  {note}: {intensity:.2f}")

OSError: cannot load library 'libsndfile.dylib': dlopen(libsndfile.dylib, 0x0002): Library not loaded: @rpath/libvorbis.0.4.9.dylib
  Referenced from: <24BBD1E7-0CB4-3E0D-A8AB-79D2B575429D> /Users/shubhan/anaconda3/envs/musichead/lib/libsndfile.1.0.37.dylib
  Reason: tried: '/Users/shubhan/anaconda3/envs/musichead/lib/libvorbis.0.4.9.dylib' (no such file), '/Users/shubhan/anaconda3/envs/musichead/lib/libvorbis.0.4.9.dylib' (no such file), '/Users/shubhan/anaconda3/envs/musichead/lib/python3.12/site-packages/../../libvorbis.0.4.9.dylib' (no such file), '/Users/shubhan/anaconda3/envs/musichead/bin/../lib/libvorbis.0.4.9.dylib' (no such file), '/usr/local/lib/libvorbis.0.4.9.dylib' (no such file), '/usr/lib/libvorbis.0.4.9.dylib' (no such file, not in dyld cache)

In [4]:
import librosa
import numpy as np

print("--- Extracting Advanced Low & Mid-Level Features ---")

# 1. Timbre via MFCCs (Mel-Frequency Cepstral Coefficients)
# Standard practice is to extract 13 or 20 coefficients. 
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
mean_mfccs = np.mean(mfccs, axis=1) # Average timbre profile of the track
print(f"MFCCs Shape (Features x Frames): {mfccs.shape}")

# 2. Brightness via Spectral Centroid
spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)
mean_centroid = np.mean(spectral_centroids)
print(f"Mean Spectral Centroid (Brightness): {mean_centroid:.2f} Hz")

# 3. Structural Change via Spectral Flux (Onset Strength)
# In Librosa, spectral flux is captured via the onset strength envelope
spectral_flux = librosa.onset.onset_strength(y=y, sr=sr)
mean_flux = np.mean(spectral_flux)
print(f"Mean Spectral Flux (Rhythmic Energy Changes): {mean_flux:.2f}")

# 4. Percussiveness via Zero-Crossing Rate (ZCR)
zcr = librosa.feature.zero_crossing_rate(y)
mean_zcr = np.mean(zcr)
print(f"Mean Zero-Crossing Rate: {mean_zcr:.4f}")

# 5. Rhythmic Separation: Harmonic vs. Percussive (HPSS)
# This splits vocals/leads from drums! Great for checking instrumentation.
y_harmonic, y_percussive = librosa.effects.hpss(y)
harmonic_energy = np.sum(y_harmonic ** 2)
percussive_energy = np.sum(y_percussive ** 2)
ratio = percussive_energy / (harmonic_energy + percussive_energy)
print(f"Percussive-to-Harmonic Ratio: {ratio * 100:.1f}%")

--- Extracting Advanced Low & Mid-Level Features ---
MFCCs Shape (Features x Frames): (13, 8571)
Mean Spectral Centroid (Brightness): 2595.13 Hz
Mean Spectral Flux (Rhythmic Energy Changes): 1.38
Mean Zero-Crossing Rate: 0.0979
Percussive-to-Harmonic Ratio: 20.2%


In [5]:
import librosa
import numpy as np

# 2. Extract the exact frames where beats occur
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)

# 3. Compute the raw Constant-Q Chromagram (CQT is sharper for musical notes than STFT)
chroma = librosa.feature.chroma_cqt(y=y, sr=sr)

# 4. Synchronize the chromagram across the beat frames
# We use the median value within each beat segment to eliminate noise transient anomalies
beat_chroma = librosa.util.sync(chroma, beat_frames, aggregate=np.median)

print(f"Total Beats Tracked: {beat_chroma.shape[1]}")
print(f"Matrix Shape (12 notes x N beats): {beat_chroma.shape}")

# Example: Print the note footprint of the first 4 beats of the song
for beat_idx in range(401, 405):
    print(f"\nBeat {beat_idx + 1} Note Footprint:")
    # Find the dominant pitch classes on this beat
    for note_idx, intensity in enumerate(beat_chroma[:, beat_idx]):
        pitch_classes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        if intensity > 0.6: # Filter for highly present frequencies
            print(f"  Active Pitch: {pitch_classes[note_idx]} (Intensity: {intensity:.2f})")

Total Beats Tracked: 512
Matrix Shape (12 notes x N beats): (12, 512)

Beat 402 Note Footprint:
  Active Pitch: F (Intensity: 1.00)

Beat 403 Note Footprint:
  Active Pitch: F (Intensity: 1.00)

Beat 404 Note Footprint:
  Active Pitch: F (Intensity: 1.00)

Beat 405 Note Footprint:
  Active Pitch: G (Intensity: 0.60)
  Active Pitch: G# (Intensity: 1.00)


In [6]:
beat_chroma.mean(axis=1)

array([0.57342863, 0.23649995, 0.19543536, 0.20133452, 0.24081782,
       0.66794133, 0.27330324, 0.3133484 , 0.21891277, 0.20271336,
       0.18093964, 0.15346946], dtype=float32)

In [2]:
import essentia.standard as es

# Initialize the master music feature extractor
# We ask it to compute the standard mean statistics for all domains
extractor = es.MusicExtractor(
    lowlevelStats=['mean'], 
    rhythmStats=['mean'], 
    tonalStats=['mean']
)

# Run the analyzer over your track
features, features_frames = extractor("audio_data/STOLE_YA_FLOW_ASAP_ROCKY.mp3")

print("--- Essentia Analysis Results ---")
print(f"Estimated BPM: {features['rhythm.bpm']:.2f}")

# Tonal DNA
print(f"Estimated Key: {features['tonal.key_edma.key']}")
print(f"Estimated Scale: {features['tonal.key_edma.scale']}")

# Time signature (meter) properties 
# Essentia computes danceability and beat grids to evaluate the underlying meter
print(f"Danceability Score: {features['rhythm.danceability']:.2f}")

2026-05-25 00:44:54.333367: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


--- Essentia Analysis Results ---
Estimated BPM: 161.64
Estimated Key: F
Estimated Scale: major
Danceability Score: 1.26


[   INFO   ] MusicExtractor: Read metadata
[   INFO   ] MusicExtractor: Compute md5 audio hash, codec, length, and EBU 128 loudness
[   INFO   ] MusicExtractor: Replay gain
[   INFO   ] MusicExtractor: Compute audio features
[   INFO   ] MusicExtractor: Compute aggregation
[   INFO   ] All done


In [17]:
import os
import librosa
import numpy as np
import pandas as pd
import essentia.standard as es

def extract_features(track_name, original_audio_path, demucs_stems_path=None):
    """
    Extracts Low, Mid, High, and Production features into a flat dictionary
    suitable for a Pandas DataFrame and Latent Space distance calculations.
    """
    # Initialize our flat feature row
    features = {'track_name': track_name}
    
    print(f"Processing: {track_name}...")
    
    # ==========================================
    # 1. LOAD AUDIO (LIBROSA)
    # ==========================================
    # Load as mono for standard spectral analysis, and stereo for spatial analysis
    y_mono, sr = librosa.load(original_audio_path, mono=True)
    y_stereo, _ = librosa.load(original_audio_path, mono=False)
    
    # ==========================================
    # 2. LOW-LEVEL (Timbre, Brightness, Texture)
    # ==========================================
    # MFCCs (Timbre) - We take the mean and variance of the first 13 coefficients
    mfccs = librosa.feature.mfcc(y=y_mono, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc_{i+1}_mean'] = np.mean(mfccs[i])
        features[f'mfcc_{i+1}_var'] = np.var(mfccs[i])
        
    # Spectral Centroid (Brightness)
    centroids = librosa.feature.spectral_centroid(y=y_mono, sr=sr)
    features['spectral_centroid_mean'] = np.mean(centroids)
    
    # Spectral Flux (Onset Strength)
    flux = librosa.onset.onset_strength(y=y_mono, sr=sr)
    features['spectral_flux_mean'] = np.mean(flux)
    
    # Zero-Crossing Rate (Noisiness/Percussiveness)
    zcr = librosa.feature.zero_crossing_rate(y_mono)
    features['zcr_mean'] = np.mean(zcr)

    # ==========================================
    # 3. MID-LEVEL (Harmony & Rhythm)
    # ==========================================
    # Chromagram (Key/Chord DNA) - Mean intensities of the 12 pitch classes
    chroma = librosa.feature.chroma_cqt(y=y_mono, sr=sr)
    pitch_classes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    for i, note in enumerate(pitch_classes):
        features[f'chroma_{note}_mean'] = np.mean(chroma[i])
        
    # Essentia Extraction for Global Key, Scale, and BPM
    extractor = es.MusicExtractor(lowlevelStats=['mean'], rhythmStats=['mean'], tonalStats=['mean'])
    es_features, _ = extractor(original_audio_path)
    
    features['bpm'] = es_features['rhythm.bpm']
    features['key'] = es_features['tonal.key_edma.key']       # e.g., 'F'
    features['scale'] = es_features['tonal.key_edma.scale']   # e.g., 'major'

    # ==========================================
    # 4. HIGH-LEVEL (Vibe & Semantics)
    # ==========================================
    features['danceability'] = es_features['rhythm.danceability']
    
    # Note: True Valence/Arousal requires Essentia's SVM models. 
    # Placeholder for when you download the Essentia Gaia models
    features['valence_pred'] = 0.0 
    features['arousal_pred'] = 0.0

    # ==========================================
    # 5. PRODUCTION & MIXING (Loudness & Spatial)
    # ==========================================
    # LUFS (EBU R128 Loudness) extracted natively by Essentia
    if 'lowlevel.loudness_ebu128.integrated' in es_features.descriptorNames():
        features['lufs_loudness'] = es_features['lowlevel.loudness_ebu128.integrated']
    else:
        features['lufs_loudness'] = 0.0
    
    # Stereo Width (Side vs Mid signal energy)
    if y_stereo.ndim == 2:
        mid_signal = (y_stereo[0] + y_stereo[1]) / 2.0
        side_signal = (y_stereo[0] - y_stereo[1]) / 2.0
        features['stereo_width_ratio'] = np.sum(side_signal**2) / (np.sum(mid_signal**2) + 1e-6)
    else:
        features['stereo_width_ratio'] = 0.0 # It's a mono track

    # ==========================================
    # 6. INSTRUMENTATION PROFILES (via Demucs)
    # ==========================================
    # If you ran Demucs and isolated the stems, we calculate the energy mix ratio!
    if demucs_stems_path and os.path.exists(demucs_stems_path):
        stems = ['vocals.wav', 'drums.wav', 'bass.wav', 'other.wav']
        energies = {}
        for stem in stems:
            stem_path = os.path.join(demucs_stems_path, stem)
            if os.path.exists(stem_path):
                y_stem, _ = librosa.load(stem_path, mono=True)
                energies[stem.split('.')[0]] = np.sum(y_stem**2)
            else:
                energies[stem.split('.')[0]] = 0.0
                
        total_energy = sum(energies.values()) + 1e-6
        features['vocal_presence_ratio'] = energies['vocals'] / total_energy
        features['drum_presence_ratio'] = energies['drums'] / total_energy
        features['bass_presence_ratio'] = energies['bass'] / total_energy
    else:
        features['vocal_presence_ratio'] = None
        features['drum_presence_ratio'] = None
        features['bass_presence_ratio'] = None

    return features

# ==========================================
# RUNNING THE PIPELINE
# ==========================================
if __name__ == "__main__":
    # Create an empty list to hold your track dictionaries
    dataset = []
    
    # Example for one track:
    # Ensure you have your paths mapped correctly based on your local machine
    song_data = extract_features(
        track_name="STOLE YA FLOW",
        original_audio_path="audio_data/STOLE_YA_FLOW_ASAP_ROCKY.mp3",
        demucs_stems_path="separated/htdemucs/STOLE_YA_FLOW_ASAP_ROCKY" # Point to Demucs output folder
    )
    
    dataset.append(song_data)
    
    # Convert to Pandas DataFrame
    df = pd.DataFrame(dataset)
    print("\n--- Final Latent Space DataFrame ---")
    print(df.head())
    
    # Save to CSV for later machine learning use
    df.to_csv("music_features_dataset.csv", index=False)

Processing: STOLE YA FLOW...


[   INFO   ] MusicExtractor: Read metadata
[   INFO   ] MusicExtractor: Compute md5 audio hash, codec, length, and EBU 128 loudness
[   INFO   ] MusicExtractor: Replay gain
[   INFO   ] MusicExtractor: Compute audio features
[   INFO   ] MusicExtractor: Compute aggregation
[   INFO   ] All done



--- Final Latent Space DataFrame ---
      track_name  mfcc_1_mean   mfcc_1_var  mfcc_2_mean   mfcc_2_var  \
0  STOLE YA FLOW    -4.041018  3245.692383    87.703346  1288.880005   

   mfcc_3_mean  mfcc_3_var  mfcc_4_mean  mfcc_4_var  mfcc_5_mean  ...  key  \
0     5.409477  646.825684    24.342058  208.884811      7.83385  ...    F   

   scale  danceability  valence_pred  arousal_pred  lufs_loudness  \
0  major      1.257281           0.0           0.0      -7.615812   

   stereo_width_ratio  vocal_presence_ratio  drum_presence_ratio  \
0            0.055959              0.158675             0.035699   

   bass_presence_ratio  
0             0.699327  

[1 rows x 53 columns]


In [3]:
from essentia.standard import TensorflowPredictMusiCNN

In [6]:
from essentia.standard import MonoLoader, TensorflowPredictMusiCNN, TensorflowPredict2D

audio = MonoLoader(filename="audio_data/STOLE_YA_FLOW_ASAP_ROCKY.mp3", sampleRate=16000, resampleQuality=4)()
embedding_model = TensorflowPredictMusiCNN(graphFilename="models/msd-musicnn-1.pb", output="model/dense/BiasAdd")
embeddings = embedding_model(audio)

model = TensorflowPredict2D(graphFilename="models/muse-msd-musicnn-2.pb", output="model/Identity")
predictions = model(embeddings)

[   INFO   ] TensorflowPredict: Successfully loaded graph file: `models/msd-musicnn-1.pb`
I0000 00:00:1779695331.310800 5965508 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
[   INFO   ] TensorflowPredict: Successfully loaded graph file: `models/muse-msd-musicnn-2.pb`


In [7]:
predictions

array([[5.1502657, 4.5601625],
       [4.6203647, 4.450602 ],
       [4.6262417, 4.4308395],
       [4.9687605, 4.421076 ],
       [5.1239243, 4.4297113],
       [4.8991385, 4.3933163],
       [5.0230103, 4.4137325],
       [5.3347073, 4.533099 ],
       [5.3442974, 4.5335183],
       [5.202849 , 4.4757695],
       [5.205217 , 4.4956903],
       [5.379915 , 4.5562363],
       [5.1656423, 4.464384 ],
       [5.1392946, 4.520035 ],
       [5.324592 , 4.5984783],
       [5.5902762, 4.6282883],
       [5.491595 , 4.7246647],
       [5.4431014, 4.7422934],
       [5.3432426, 4.7596455],
       [4.9134097, 4.6751328],
       [5.352708 , 4.5927706],
       [5.314082 , 4.8123684],
       [5.393083 , 4.85205  ],
       [5.290513 , 4.6984897],
       [5.095312 , 4.6884246],
       [4.9439526, 4.6581287],
       [5.151019 , 4.732403 ],
       [5.1850553, 4.9023542],
       [5.252597 , 4.871066 ],
       [5.110041 , 4.900875 ],
       [5.2470655, 4.8474636],
       [5.346704 , 4.9193597],
       [

In [8]:
predictions.shape

(132, 2)

In [1]:
from essentia.standard import MonoLoader, TensorflowPredictEffnetDiscogs, TensorflowPredict2D

audio = MonoLoader(filename="audio_data/STOLE_YA_FLOW_ASAP_ROCKY.mp3", sampleRate=16000, resampleQuality=4)()
embedding_model = TensorflowPredictEffnetDiscogs(graphFilename="models/discogs-effnet-bs64-1.pb", output="PartitionedCall:1")
embeddings = embedding_model(audio)

model = TensorflowPredict2D(graphFilename="models/mtg_jamendo_instrument-discogs-effnet-1.pb")
predictions = model(embeddings)

2026-06-29 20:58:32.042867: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


RuntimeError: Error while configuring MonoLoader: AudioLoader: Could not open file "audio_data/STOLE_YA_FLOW_ASAP_ROCKY.mp3", error = No such file or directory

In [13]:
print(predictions.shape)

(199, 40)


In [14]:
predictions

array([[0.00919555, 0.01067947, 0.02667514, ..., 0.0084468 , 0.04123184,
        0.03957857],
       [0.00934748, 0.0080319 , 0.02870857, ..., 0.00919694, 0.04476125,
        0.04340561],
       [0.00572917, 0.00779019, 0.03460849, ..., 0.01363947, 0.04697539,
        0.04906133],
       ...,
       [0.01022477, 0.01388436, 0.04359607, ..., 0.00986987, 0.03755077,
        0.08801453],
       [0.0167436 , 0.01342165, 0.06756642, ..., 0.01551209, 0.05656133,
        0.11101054],
       [0.0142249 , 0.01015365, 0.05731253, ..., 0.01342708, 0.07363693,
        0.07366624]], dtype=float32)